# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR^2 dataset with the `mlcroissant` library, referencing entities by their `@id` according to the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
md = dataset.metadata
print(f"{md.name}: {md.description}")

# Optionally print available metadata fields
pprint.pprint(md.to_json())

## 2. Data Overview
Review available record sets, fields, and their `@id` values.
This step is crucial: all entities must be referenced by their `@id` as defined in the schema. We'll enumerate record sets (tables), their fields (columns & variables), and print their `@id`s.

In [ ]:
# List record sets by @id
record_sets = dataset.metadata.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f"  RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")

    # List fields and columns in this record set
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field name: {field.name}")
        print(f"      @id: {field.id} (type: {field.data_type})")
    print()

# Example: Print some sample records from the first record set
if record_sets:
    rs_id = record_sets[0].id
    print(f"Sample records from RecordSet @id: {rs_id}")
    for rec in dataset.records(record_set=rs_id):
        print(rec)
        break  # Print one for overview

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

We'll list record sets and use their `@id`s to dynamically extract all records, referencing fields by their `@id` in subsequent analysis.

In [ ]:
# Extract data from each record set
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for RecordSet @id: {rs_id}, shape: {df.shape}")
    print(f"Columns (@id): {df.columns.tolist()}")
    print(df.head(), '\n')

# Select a record set for further analysis
main_record_set_id = record_set_ids[0] if record_set_ids else None
# Print its columns (@id)
if main_record_set_id:
    print(f"Columns in RecordSet {main_record_set_id}: {dataframes[main_record_set_id].columns.tolist()}")
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps using `@id` references.
- Filter records based on a numeric field
- Normalize values
- Group by categorical field

You must reference all fields/columns using their `@id` as listed in the previous section.

In [ ]:
# Identify numeric and categorical fields by their @id
if main_record_set_id:
    main_rs = [rs for rs in dataset.metadata.record_sets if rs.id == main_record_set_id][0]
    numeric_fields = [field.id for field in main_rs.fields if field.data_type in ['Integer', 'Float', 'Number']]
    categorical_fields = [field.id for field in main_rs.fields if field.data_type == 'Text']
    df = dataframes[main_record_set_id]
    print(f"Numeric fields (@id): {numeric_fields}")
    print(f"Categorical fields (@id): {categorical_fields}")

    # Filter by first available numeric field
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        threshold = 50  # change as appropriate for dataset
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical field if available
        if categorical_fields:
            group_field_id = categorical_fields[0]
            if group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"Grouped filtered records by {group_field_id} (mean {numeric_field_id}):")
                print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships using fields referenced by their `@id`.

- Histogram of numeric field
- Boxplot grouped by a categorical field

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_fields:
    numeric_field_id = numeric_fields[0]
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot grouped by categorical field
    if categorical_fields:
        group_field_id = categorical_fields[0]
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Through structured extraction using `mlcroissant`, we've loaded the Clinicopathological and Molecular Characteristics dataset, referenced all entities by their `@id`, filtered and normalized key numeric fields, grouped by clinical categories, and visualized major record distributions.

Further steps could involve deeper statistical analysis, clinical stratification, or exporting processed data for downstream modeling tasks.